In [1]:
from typing import List, TypedDict
import time

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_community.embeddings import JinaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

/var/folders/1w/mwjv_0hn6vqf1xtmrdz34ml00000gn/T/ipykernel_14049/4016716716.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

In [5]:
# pip install pypdf

In [6]:

docs = (
    PyPDFLoader("./documents/book1.pdf").load() +
    PyPDFLoader("./documents/book2.pdf").load() +
    PyPDFLoader("./documents/book3.pdf").load()
)

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/FontDescriptor': IndirectObject(25, 0, 4426036368), '/LastChar': 1, '/Widths': [833], '/BaseFont': '/NJBOIP+MathematicalPi-Three', '/FirstChar': 1, '/Encoding': IndirectObject(26, 0, 4426036368), '/Type': '/Font'}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/FontDescriptor': IndirectObject(743, 0, 4426036368), '/LastChar': 2, '/Widths': [778, 778], '/BaseFont': '/NJDFHF+MSAM10', '/FirstChar': 1, '/Encoding': IndirectObject(744, 0, 4426036368), '/Type': '/Font'}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/FontDescriptor': IndirectObject(743, 0, 4426036368), '/Last

In [7]:

len(docs)

2123

In [8]:
# 2) Chunk
chunks = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150).split_documents(docs)

# 3) Clean text to avoid UnicodeEncodeError (surrogates from PDF extraction)
for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

In [11]:
print(chunks[0].page_content)
print("="*20)
print(chunks[1].page_content)

Information Science and Statistics
Series Editors:
M. Jordan
J. Kleinberg
B. Scho¨lkopf
Information Science and Statistics 
Akaike and Kitagawa: The Practice of Time Series Analysis. 
Bishop:  Pattern Recognition and Machine Learning. 
Cowell, Dawid, Lauritzen, and Spiegelhalter: Probabilistic Networks and
Expert Systems. 
Doucet, de Freitas, and Gordon: Sequential Monte Carlo Methods in Practice. 
Fine: Feedforward Neural Network Methodology. 
Hawkins and Olwell: Cumulative Sum Charts and Charting for Quality Improvement. 
Jensen: Bayesian Networks and Decision Graphs. 
Marchette: Computer Intrusion Detection and Network Monitoring:
A Statistical Viewpoint. 
Rubinstein and Kroese: The Cross-Entropy Method:  A Unified Approach to 
Combinatorial Optimization, Monte Carlo Simulation, and Machine Learning. 
Studený: Probabilistic Conditional Independence Structures.
Vapnik: The Nature of Statistical Learning Theory, Second Edition.


In [2]:
import os

JINA_API_KEY = os.getenv("JINA_API_KEY")

embedder = JinaEmbeddings(

    jina_api_key=os.getenv("JINA_API_KEY"),

    model_name="jina-embeddings-v5-omni-nano"

)

In [19]:
# import time

# BATCH_SIZE = 32
# DELAY = 5  # 5 seconds between batches

# # Create the first batch
# vector_store = FAISS.from_documents(
#     documents=chunks[:BATCH_SIZE],
#     embedding=embedder
# )

# # Process remaining batches
# for start in range(BATCH_SIZE, len(chunks), BATCH_SIZE):

#     batch = chunks[start:start + BATCH_SIZE]

#     vector_store.add_documents(batch)

#     print(f"Processed {min(start + BATCH_SIZE, len(chunks))}/{len(chunks)} chunks")

#     # Wait 5 seconds before processing the next batch
#     time.sleep(DELAY)

# # Save database
# vector_store.save_local("faiss_db")

# print("FAISS database saved successfully!")

In [17]:
# 2. Load existing FAISS database

vector_store = FAISS.load_local(

    "faiss_db",

    embeddings=embedder,

    allow_dangerous_deserialization=True

)
# 3. Create retriever
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

In [18]:
MODEL_NAME = "gemini-3.5-flash-lite"

model = init_chat_model(
    MODEL_NAME,
    model_provider="google_genai",
    temperature=0,
)

In [24]:
# 2. Create retrieval tool
from langchain.tools import tool
from langchain.agents import create_agent

@tool
def retrieve_documents(query: str) -> str:

    """Search the FAISS vector database and retrieve relevant document chunks."""

    # Get FAISS relevance scores (0.0 = least relevant, 1.0 = most relevant).
    scored_docs = vector_store.similarity_search_with_relevance_scores(
        query, k=4
    )

    if not scored_docs:

        return "No relevant documents found."

    context = "\n\n".join(

        (
            f"Document {index} | Relevance: {max(1, min(10, round(score * 10)))} / 10\n"
            f"{doc.page_content}"
        )
        for index, (doc, score) in enumerate(scored_docs, start=1)

    )

    return context

# 3. Create LangChain agent

agent = create_agent(

    model=model,

    tools=[retrieve_documents],

    system_prompt="""

    You are a document-based question-answering assistant.

    Follow these instructions:

    1. Always use the retrieve_documents tool to search

       the knowledge base before answering.

    2. Use only the retrieved document context to answer.

    3. Do not use your own knowledge or external information.

    4. If the answer is not available in the retrieved

       documents, say "I don't know based on the provided documents."

    5. Provide clear and accurate answers.

    """

)



In [22]:
response = agent.invoke({

    "messages": [

        {

            "role": "user",


            "content": "WHat is a transformer in deep learning?"

        }

    ]

})

/opt/anaconda3/envs/LandGraph/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/opt/anaconda3/envs/LandGraph/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/opt/anaconda3/envs/LandGraph/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/opt/anaconda3/envs/LandGraph/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the samplin

In [23]:
print(response["messages"][-1].content)

[{'type': 'text', 'text': 'In the context of data preparation and machine learning APIs (such as Scikit-Learn), a **transformer** is an estimator (such as an imputer) that can transform a dataset. \n\nKey characteristics of transformers include:\n* They perform transformations on a dataset via the `transform()` method, which takes the dataset as a parameter and returns the transformed dataset.\n* Their transformations generally rely on parameters learned during a training or estimation process.\n* They often provide a convenience method called `fit_transform()` to combine estimating parameters and transforming the dataset in one step.\n\n*(Note: If you are asking about the Transformer neural network architecture used in deep learning for natural language processing, that specific information is not available in the provided documents.)*', 'extras': {'signature': 'EmAKXgFpFH0TugvuKHExmPMbanUz9bdzAMQb38FNOqCkz8dsZRW+vbufAf7vghDc+QCj1gjOcYu64SyU91knvoYcPn86fxJhPd1QzsLFziGSpbhUE3+wwYyXVquy

In [25]:
for message in response["messages"]:
    print("Message type:", message.type)
    print("Content:", message.content)
    print("*" * 100)

Message type: human
Content: WHat is a transformer in deep learning?
****************************************************************************************************
Message type: ai
Content: []
****************************************************************************************************
Message type: tool
Content: All transformers also have a convenience method called fit_transform() 
Prepare the Data for Machine Learning Algorithms | 61
Download from finelybook www.finelybook.com

DeepLearning
IanGoodfellow
YoshuaBengio
AaronCourville

training and visualizing, 167-169
decoder, 412
deconvolutional layer, 
376
deep autoencoders (see stacked autoencoders)
deep belief networks (DBNs), 13, 519-521
Deep Learning, 437
(see also Reinforcement Learning; Tensor‐
Flow)
about, xiii, xvi
libraries, 230-231
deep neural networks (DNNs), 261, 275-312
(see also Multi-Layer Perceptrons (MLP))
faster optimizers for, 293-302
regularization, 302-310
reusing pretrained layers, 286-293
training